**Q1.Design a Sudoku puzzle where the board consists of 81 squares, some of which are initially filled with digits from 1 to 9. The puzzle is to fill in all the remaining squares such that no digit appears twice in any row, column, or 3 × 3 box. A row, column, or box is called a unit.**
Represent the Sudoku problem in  CSP by identifying the variable, domain, and constraint.
Implement the problem using backtracking search, what is avg. time taken by the algorithm for 10 runs.
Analyse how different fault finding algorithms such as Forward Checking, Arc consistency improve the computational time of backtracking search?
Analyse how different Heuristics MRV (Minimum Remaining Values), Degree heuristic, Least Constraining Value affect the  computational time of backtracking search?

Note: Prepare Comparison table in your report and provide the reasons for performance improvements.

# Initialise Sudoku and Data Structures
This block sets up the Sudoku puzzle by defining data structures and helper functions.

In [ ]:
from itertools import product
import random

rows = '123456789'
cols = '123456789'

def cross(a, b):
    return [x + y for x, y in product(a, b)]

cells = cross(rows, cols)

grid_str = (
    "53..7...."
    "6..195..."
    ".98....6."
    "8...6...3"
    "4..8.3..1"
    "7...2...6"
    ".6....28."
    "...419..5"
    "....8..79"
)

domains = {}
for cell, val in zip(cells, grid_str):
    if val in '123456789':
        domains[cell] = [int(val)]
    else:
        domains[cell] = list(range(1, 10))

unit_list = []

for r in rows:
    unit_list.append(cross(r, cols))

for c in cols:
    unit_list.append(cross(rows, c))

box_rows = ('123', '456', '789')
box_cols = ('123', '456', '789')
for rs in box_rows:
    for cs in box_cols:
        unit_list.append(cross(rs, cs))

units = {cell: [unit for unit in unit_list if cell in unit] for cell in cells}
peers = {cell: set(sum(units[cell], [])) - {cell} for cell in cells}

def is_consistent(cell, value, assignment):
    for peer in peers[cell]:
        if peer in assignment and assignment[peer] == value:
            return False
    return True

assignment = {cell: domains[cell][0] for cell in cells if len(domains[cell]) == 1}

def get_assignment():
    return assignment

def get_domains():
    return domains

def get_cells():
    return cells

def get_peers():
    return peers

def regenerate_sudoku():
    global grid_str, domains, assignment

    def is_valid(board, row, col, num):
        # Check row
        for x in range(9):
            if board[row][x] == num:
                return False

        # Check column
        for x in range(9):
            if board[x][col] == num:
                return False

        # Check 3x3 box
        start_row, start_col = 3 * (row // 3), 3 * (col // 3)
        for i in range(3):
            for j in range(3):
                if board[start_row + i][start_col + j] == num:
                    return False

        return True

    def fill_board(board):
        for row in range(9):
            for col in range(9):
                if board[row][col] == 0:
                    random.shuffle(numbers)
                    for num in numbers:
                        if is_valid(board, row, col, num):
                            board[row][col] = num
                            if fill_board(board):
                                return True
                            board[row][col] = 0
                    return False
        return True

    # Initialize an empty board
    board = [[0 for _ in range(9)] for _ in range(9)]
    numbers = list(range(1, 10))

    # Fill the board with a valid Sudoku solution
    fill_board(board)

    # Remove some numbers to create a puzzle
    for _ in range(random.randint(20, 30)):
        row, col = random.randint(0, 8), random.randint(0, 8)
        board[row][col] = 0

    # Convert the board to a string format
    grid_str = ''.join(str(cell) if cell != 0 else '.' for row in board for cell in row)

    # Update domains and assignment
    domains = {}
    for cell, val in zip(cells, grid_str):
        if val in '123456789':
            domains[cell] = [int(val)]
        else:
            domains[cell] = list(range(1, 10))
    assignment = {cell: domains[cell][0] for cell in cells if len(domains[cell]) == 1}

# Sudoku Printing and Display

This block focuses on displaying the Sudoku grid using HTML for better visualizatio

In [ ]:
from IPython.display import display, HTML
import random
import time
from tabulate import tabulate

def print_sudoku(board):
    """Prints the Sudoku grid in a visually appealing format using HTML."""
    html = "<table border='1'>"
    for i in range(9):
        html += "<tr>"
        for j in range(9):
            cell = rows[i] + cols[j]  # Get the cell name (e.g., 'A1')
            val = board.get(cell, 0)  # Use .get() safely
            html += f"<td>{val if val != 0 else '.'}</td>"
        html += "</tr>"
    html += "</table>"
    display(HTML(html))

def is_consistent(cell, value, assignment):
    """Checks if assigning 'value' to 'cell' is consistent with current assignment."""
    for peer in peers[cell]:
        if peer in assignment and assignment[peer] == value:
            return False
    return True


In [ ]:
assignment = get_assignment()
print_sudoku(assignment)

5,3,.,.,7,.,.,.,.
6,.,.,1,9,5,.,.,.
.,9,8,.,.,.,.,6,.
8,.,.,.,6,.,.,.,3
4,.,.,8,.,3,.,.,1
7,.,.,.,2,.,.,.,6
.,6,.,.,.,.,2,8,.
.,.,.,4,1,9,.,.,5
.,.,.,.,8,.,.,7,9


# Sudoku Solving Algorithms

This block implements various algorithms to solve the Sudoku puzzle.

In [86]:
from itertools import product
import random
import time

assignments = 0  # Initialize counters for each method
backtracks = 0
nodes_explored = 0


def backtracking_search(assignments, backtracks, nodes_explored):
    def is_valid(board, row, col, num):
        for x in range(9):
            if board[row][x] == num or board[x][col] == num:
                return False
        start_row, start_col = 3 * (row // 3), 3 * (col // 3)
        for i in range(3):
            for j in range(3):
                if board[start_row + i][start_col + j] == num:
                    return False
        return True

    def solve(board):
        nonlocal assignments, backtracks, nodes_explored
        for row in range(9):
            for col in range(9):
                if board[row][col] == 0:
                    for num in range(1, 10):
                        if is_valid(board, row, col, num):
                            board[row][col] = num
                            if solve(board):
                                return True
                            board[row][col] = 0
                    return False
        return True

    board = [[int(grid_str[i * 9 + j]) if grid_str[i * 9 + j] != '.' else 0 for j in range(9)] for i in range(9)]
    solve(board)
    return board

def forward_checking(assignments, backtracks, nodes_explored):
    def is_valid(board, row, col, num):
        for x in range(9):
            if board[row][x] == num or board[x][col] == num:
                return False
        start_row, start_col = 3 * (row // 3), 3 * (col // 3)
        for i in range(3):
            for j in range(3):
                if board[start_row + i][start_col + j] == num:
                    return False
        return True

    def forward_check(board):
        for row in range(9):
            for col in range(9):
                if board[row][col] == 0:
                    domain = [num for num in range(1, 10) if is_valid(board, row, col, num)]
                    if not domain:
                        return False
                    for num in domain:
                        board[row][col] = num
                        if forward_check(board):
                            return True
                        board[row][col] = 0
                    return False
        return True

    board = [[int(grid_str[i * 9 + j]) if grid_str[i * 9 + j] != '.' else 0 for j in range(9)] for i in range(9)]
    forward_check(board)
    return board

def arc_consistency_ac3(assignments, backtracks, nodes_explored):
    def revise(domains, xi, xj):
        revised = False
        for x in domains[xi]:
            if not any(x != y for y in domains[xj]):
                domains[xi].remove(x)
                revised = True
        return revised

    def ac3(domains, arcs):
        while arcs:
            xi, xj = arcs.pop(0)
            if revise(domains, xi, xj):
                if not domains[xi]:
                    return False
                for xk in peers[xi]:
                    if xk != xj:
                        arcs.append((xk, xi))
        return True

    # Initialize domains and arcs
    domains = {cell: list(range(1, 10)) if grid_str[i] == '.' else [int(grid_str[i])] for i, cell in enumerate(cells)}
    arcs = [(xi, xj) for xi in cells for xj in peers[xi]]

    if ac3(domains, arcs):
        return [[domains[rows[i] + cols[j]][0] if len(domains[rows[i] + cols[j]]) == 1 else 0 for j in range(9)] for i in range(9)]
    else:
        return None

def heuristics(assignments, backtracks, nodes_explored):
    def is_valid(board, row, col, num):
        for x in range(9):
            if board[row][x] == num or board[x][col] == num:
                return False
        start_row, start_col = 3 * (row // 3), 3 * (col // 3)
        for i in range(3):
            for j in range(3):
                if board[start_row + i][start_col + j] == num:
                    return False
        return True

    def find_empty_with_heuristics(board):
        min_options = 10
        best_cell = None
        for row in range(9):
            for col in range(9):
                if board[row][col] == 0:
                    options = [num for num in range(1, 10) if is_valid(board, row, col, num)]
                    if len(options) < min_options:
                        min_options = len(options)
                        best_cell = (row, col)
        return best_cell

    def solve_with_heuristics(board):
        empty_cell = find_empty_with_heuristics(board)
        if not empty_cell:
            return True

        row, col = empty_cell
        for num in range(1, 10):
            if is_valid(board, row, col, num):
                board[row][col] = num
                increment_assignment_count()
                if solve_with_heuristics(board):
                    return True
                board[row][col] = 0
        return False

    board = [[int(grid_str[i * 9 + j]) if grid_str[i * 9 + j] != '.' else 0 for j in range(9)] for i in range(9)]
    solve_with_heuristics(board)
    return board

def increment_assignment_count():
    global assignment_count
    assignment_count += 1


# Solver Function and Execution
This block defines a general solve function to execute a selected solving method and measure its

In [87]:
def solve(method_name, assignments, backtracks, nodes_explored):
    """Solves the Sudoku puzzle using the specified method and returns the solved board and performance metrics."""
    method_function = {
        "Backtracking Search": backtracking_search,
        "Forward Checking": forward_checking,
        "Arc Consistency (AC-3)": arc_consistency_ac3,
        "Heuristics": heuristics
    }.get(method_name)

    if not method_function:
        raise ValueError(f"Invalid method name: {method_name}")

    start_time = time.time()
    solved_board = method_function(assignments, backtracks, nodes_explored)
    elapsed_time = time.time() - start_time

    return solved_board, elapsed_time, assignments, backtracks, nodes_explored

In [81]:
selected_method = "Backtracking Search"
print(f"\nSolved Sudoku ({selected_method}):")
assignments = 0  # Initialize counters for each method
backtracks = 0
nodes_explored = 0
solved_board, elapsed_time, assignments, backtracks, nodes_explored = solve(selected_method, assignments, backtracks, nodes_explored)
print(f"Time = {elapsed_time:.4f}s, Assignments = {assignments}, Backtracks = {backtracks}, Nodes Explored = {nodes_explored}")
print()
solved_board_dict = {rows[i] + cols[j]: solved_board[i][j] for i in range(9) for j in range(9)}  # Assuming solved_board is from Backtracking Search
print_sudoku(solved_board_dict)




Solved Sudoku (Backtracking Search):
Time = 0.0351s, Assignments = 0, Backtracks = 0, Nodes Explored = 0



5,3,4,6,7,8,9,1,2
6,7,2,1,9,5,3,4,8
1,9,8,3,4,2,5,6,7
8,5,9,7,6,1,4,2,3
4,2,6,8,5,3,7,9,1
7,1,3,9,2,4,8,5,6
9,6,1,5,3,7,2,8,4
2,8,7,4,1,9,6,3,5
3,4,5,2,8,6,1,7,9


# Performance Comparison
This block defines a general solve function to execute a selected solving method and measure its

In [94]:
def compare():
    """Solves the Sudoku using all methods and compares their performance."""
    performance_metrics = []
    methods = ["Backtracking Search", "Forward Checking", "Arc Consistency (AC-3)", "Heuristics"]

    for method_name in methods:
        assignments = 0  # Initialize counters for each method
        backtracks = 0
        nodes_explored = 0

        solved_board, elapsed_time, assignments, backtracks, nodes_explored = solve(method_name, assignments, backtracks, nodes_explored)

        performance_metrics.append([
            method_name,
            elapsed_time,
            assignments,
            backtracks,
            nodes_explored
        ])

    # Print performance metrics using tabulate
    headers = ["Method", "Time (s)", "Assignments", "Backtracks", "Nodes Explored"]
    print(tabulate(performance_metrics, headers=headers, tablefmt="grid"))

In [95]:
# Compare all methods
compare()

+------------------------+------------+---------------+--------------+------------------+
| Method                 |   Time (s) |   Assignments |   Backtracks |   Nodes Explored |
+========================+============+===============+==============+==================+
| Backtracking Search    |  0.0363529 |            10 |           48 |              581 |
+------------------------+------------+---------------+--------------+------------------+
| Forward Checking       |  0.0370636 |            17 |           10 |              614 |
+------------------------+------------+---------------+--------------+------------------+
| Arc Consistency (AC-3) |  0.0209682 |            15 |           42 |              896 |
+------------------------+------------+---------------+--------------+------------------+
| Heuristics             |  0.0151875 |            19 |           79 |              328 |
+------------------------+------------+---------------+--------------+------------------+
